# R515B Soğutucusu için STGP‑EF Tabanlı Özellik Zenginleştirme ve Regresyon Performans Analizi

Bu çalışma, R515B termodinamik soğutucu akışkanının doymuş ve kızgın fazlarındaki
durumunu tanımlayan veri setlerinde:

- Girdi öznitelik sayısının sınırlı olmasından kaynaklanan temsil gücü kısıtını azaltmak,
- **Sembolik Genetik Programlama (STGP)** ve **Evrimsel Orman (EF)** bileşenlerinden oluşan
  **STGP‑EF hibrit yöntemi** ile girdi uzayını zenginleştirmek,
- Bu zenginleştirmenin **11 farklı regresyon algoritmasının** tahmin performansına etkisini
  senaryo bazında değerlendirmek

amacıyla gerçekleştirilmiştir.

Ayrıca çalışmada, aşağıdaki araştırma sorularına yanıt aranmaktadır:

1. Base (yalnızca orijinal girdiler) ve STGP‑EF (orijinal + türetilmiş öznitelikler) senaryoları
   kıyaslandığında, her (hedef, algoritma) çifti için Test R² bakımından toplamda kaç kazanım
   elde edilmektedir ve ortalama katkı nedir?

2. STGP‑EF yaklaşımı, ortalamada hangi regresyon algoritmalarının başarımına ne ölçüde katkı
   sağlamaktadır?

3. STGP‑EF, ortalamada hangi hedef (çıktı) büyüklüklerinin tahmin başarımını ne kadar
   iyileştirmektedir?

4. STGP‑EF ile üretilen hibrit özniteliklerin:
   - Bireysel ve
   - Toplu

   başarıma katkıları algoritma ve çıktı bazında nasıl dağılmaktadır?

5. Üretilen öznitelikler ve bunlara dayalı tahminler, doymuş ve kızgın faz için temel
   termodinamik yasalara ne ölçüde uyum göstermektedir?

Aşağıda, bu sorulara yanıt vermek üzere tasarlanmış analiz akışı sunulmaktadır.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")


from Functions import (
    SATURATED_INPUTS, SATURATED_OUTPUTS,
    SUPERHEATED_INPUTS, SUPERHEATED_OUTPUTS,
    run_saturated_hybrid_analysis,
    run_superheated_hybrid_analysis,
    build_results_table,
    analyze_stgp_ef_gains,
    summarize_hybrid_contributions_for_all_targets,
    check_saturated_pointwise_constraints,
    check_saturated_monotonicity,
    check_superheated_monotonicity
)



c:\Users\SGUZEY\Documents\Yüksek Lisans Dersleri\Veri Mühendisliği\Data_Engineering_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-21 22:24:11,745 - INFO - Loading faiss with AVX2 support.
2026-05-21 22:24:11,774 - INFO - Successfully loaded faiss with AVX2 support.


## 1. Veri Setlerinin Yüklenmesi

Bu bölümde, R515B soğutucu akışkanına ait:

- Doymuş faz (saturated) veri seti ve
- Kızgın faz (superheated) veri seti

yüklenmekte ve temel yapıları incelenmektedir.


In [2]:
# Veri seti yolları (kendi çalışma ortamınıza göre güncelleyin)

saturated_path = "R515B_Ozellikler/R515B_Doymus_Ozellikleri.csv"
superheated_path = "R515B_Ozellikler/R515B_Kizgin_Buhar.csv"

df_saturated = pd.read_csv(saturated_path)
df_superheated = pd.read_csv(superheated_path)

print("Doymuş faz veri seti boyutu:", df_saturated.shape)
print("Kızgın faz veri seti boyutu:", df_superheated.shape)


Doymuş faz veri seti boyutu: (136, 8)
Kızgın faz veri seti boyutu: (578, 5)


In [3]:
# Doymuş faz veri setinin ilk satırları
df_saturated.head()


,T(girdi),P(çıktı),v sıvı (çıktı),v buhar (çıktı),h sıvı (çıktı),h buhar (çıktı),s sıvı (çıktı),s buhar (çıktı)
0,-30,61.2210,0.0007,0.2726,2.7282,198.2000,0.0446,0.8486
1,-29,64.2350,0.0007,0.2606,3.9422,198.8900,0.0496,0.8481
2,-28,67.3670,0.0007,0.2492,5.1587,199.5800,0.0546,0.8476
3,-27,70.6200,0.0007,0.2385,6.3775,200.2700,0.0595,0.8472
4,-26,73.9960,0.0008,0.2282,7.5988,200.9500,0.0645,0.8468


In [4]:
# Kızgın faz veri setinin ilk satırları
df_superheated.head()


,T (girdi),P (girdi),v (çıktı),h (çıktı),s (çıktı)
0,-30,50,0.3357,198.5500,0.8639
1,-25,50,0.3433,202.4500,0.8798
2,-20,50,0.3508,206.4000,0.8956
3,-15,50,0.3582,210.4000,0.9112
4,-10,50,0.3656,214.4300,0.9267


## 2. STGP‑EF Hibrit Analizinin Çalıştırılması

Bu aşamada, yalnızca **Base** ve **STGP‑EF** senaryolarını içeren hibrit analiz
fonksiyonları kullanılacaktır:

- Doymuş faz için: `run_saturated_hybrid_analysis`
- Kızgın faz için: `run_superheated_hybrid_analysis`

Her bir hedef değişken ve her bir regresyon algoritması için:

- Base senaryosunda,
- STGP‑EF senaryosunda

elde edilen performans metrikleri (özellikle Test R²) kayıt altına alınacaktır.


In [5]:
# Doymuş faz için hibrit analiz (Base vs STGP‑EF)

sat_hybrid_results = run_saturated_hybrid_analysis(df_saturated)

# Sonuçları uzun formata dönüştür
sat_results_df = build_results_table(sat_hybrid_results)
sat_results_df['Dataset'] = 'R515B_saturated'

sat_results_df.head()



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DOYMUŞ BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: P(çıktı)  |  Input: ['T(girdi)']
  [1/2] Base training...


2026-05-21 22:25:13,107 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [2/2] STGP-EF training...


2026-05-21 22:25:20,102 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:25:20,103 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 22:25:20,103 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:25:20,104 - INFO -   STGP_00: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,105 - INFO -   STGP_01: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,105 - INFO -   STGP_02: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,106 - INFO -   STGP_03: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,106 - INFO -   STGP_04: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,107 - INFO -   STGP_05: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,108 - INFO -   STGP_06: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,108 - INFO -   STGP_07: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 22:25:20,109 - INFO -   STGP_08: (

  Completed.


  Target: v sıvı (çıktı)  |  Input: ['T(girdi)']
  [1/2] Base training...


...

2026-05-21 22:49:59,378 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [2/2] STGP-EF training...


2026-05-21 22:50:08,280 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:50:08,282 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 22:50:08,282 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:50:08,282 - INFO -   STGP_00: (((((x0 * x0) + (0.888 + (((x0 * x0) + (0.888 + (0.888 + (0.888 + x0)))) + x0))) + (0.888 + 0.678)) + (0.888 + x0)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-21 22:50:08,282 - INFO -   STGP_01: ((((0.888 + x0) + ((((x0 * x0) + (0.888 + (0.888 + x0))) + 0.888) + ((((x0 * x0) + (0.888 + x0)) + 0.888) + 0.888))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-21 22:50:08,282 - INFO -   STGP_02: ((((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) + (0.888 + (0.888 + x0))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-2

  Completed.



,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE,Dataset
0,P(çıktı),Base,AdaBoost,0.9988,0.9986,0.0343,0.0377,0.0242,0.0298,0.0509,0.0367,R515B_saturated
1,P(çıktı),Base,CatBoost,0.9998,0.9995,0.0129,0.0229,0.0101,0.0167,0.0337,0.0307,R515B_saturated
2,P(çıktı),Base,DART,0.9544,0.9610,0.2136,0.2011,0.1200,0.1150,0.1497,0.1097,R515B_saturated
3,P(çıktı),Base,EF,1.0000,1.0000,0.0000,0.0032,0.0000,0.0022,0.0000,0.0036,R515B_saturated
4,P(çıktı),Base,ET,1.0000,1.0000,0.0000,0.0025,0.0000,0.0017,0.0000,0.0024,R515B_saturated


In [6]:
# Kızgın faz için hibrit analiz (Base vs STGP‑EF)

sup_hybrid_results = run_superheated_hybrid_analysis(df_superheated)

sup_results_df = build_results_table(sup_hybrid_results)
sup_results_df['Dataset'] = 'R515B_superheated'

sup_results_df.head()



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  KIZGIN BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: v (çıktı)  |  Input: ['T (girdi)', 'P (girdi)']
  [1/2] Base training...


2026-05-21 22:54:24,357 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [2/2] STGP-EF training...


2026-05-21 22:54:32,971 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:54:32,971 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 22:54:32,971 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 22:54:32,971 - INFO -   STGP_00: (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0)))) - ((x1 * x1) - x0))) * (x1 * x1)))
2026-05-21 22:54:32,971 - INFO -   STGP_01: (((x1 * x1) - ((x1 + x1) + x1)) * ((x1 + (((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0))))) * (x1 * x1)))
2026-05-21 22:54:32,971 - INFO -   STGP_02: (((x1 * x1) - (x1 + (x1 + x1))) * ((x1 + (((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - (x1 + x1)) * ((((x1 * x1) - x1) * ((x1 * x1) - (x1 + x1))) 

  Completed.


  Target: h (çıktı)  |  Input: ['T (girdi)', 'P (girdi)']
  [1/2] Base training...


...

2026-05-21 23:05:14,450 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [2/2] STGP-EF training...


2026-05-21 23:05:22,165 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:05:22,165 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 23:05:22,165 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:05:22,181 - INFO -   STGP_00: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-21 23:05:22,181 - INFO -   STGP_01: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-21 23:05:22,183 - INFO -   STGP_02: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-21 23:05:22,183 - INFO -   STGP_03: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-21 23:05:22,183 - INFO -   STGP_04: (x1 - ((x1 * (x0 * 0.31)) + x0))
2026-05-21 23:05:22,185 - INFO -   STGP_05: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-21 23:05:22,185 - INFO -   STGP_06: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-21 23:05:22,185 - INFO -   STGP_07: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-21 23:05:22,185 - INFO -   STGP_08: (((x1 * (x0 * 0.31)) + x0) - x1)


  Completed.



,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE,Dataset
0,v (çıktı),Base,AdaBoost,0.9941,0.9822,0.0770,0.1095,0.0633,0.0829,0.3407,0.2540,R515B_superheated
1,v (çıktı),Base,CatBoost,0.9999,0.9979,0.0089,0.0379,0.0060,0.0148,0.0331,0.0372,R515B_superheated
2,v (çıktı),Base,DART,0.9820,0.9908,0.1342,0.0786,0.0632,0.0516,0.1270,0.1050,R515B_superheated
3,v (çıktı),Base,EF,1.0000,0.9998,0.0004,0.0116,0.0002,0.0047,0.0007,0.0150,R515B_superheated
4,v (çıktı),Base,ET,1.0000,0.9998,0.0000,0.0129,0.0000,0.0046,0.0000,0.0140,R515B_superheated


## 3. Base ve STGP‑EF Senaryolarının Karşılaştırılması

Bu bölümde, `analyze_stgp_ef_gains` fonksiyonu kullanılarak:

- Her bir (Target, Algorithm) çifti için
  - Base senaryosundaki Test R² değeri,
  - STGP‑EF senaryosundaki Test R² değeri

karşılaştırılacak ve:

- Toplam kaç durumda iyileşme sağlandığı,
- Ortalama Test R² kazanımı,

senaryo, algoritma ve hedef bazında özetlenecektir.


In [7]:
# Doymuş faz: Base vs STGP‑EF karşılaştırması

sat_gain_analysis = analyze_stgp_ef_gains(sat_results_df)

sat_pivot = sat_gain_analysis['pivot']
sat_scenario_summary = sat_gain_analysis['scenario_summary']
sat_algo_summary = sat_gain_analysis['algo_summary']
sat_target_summary = sat_gain_analysis['target_summary']

print("Doymuş faz - Genel senaryo özeti (Base vs STGP‑EF):")
sat_scenario_summary


Doymuş faz - Genel senaryo özeti (Base vs STGP‑EF):


{'total_cases': 77,
 'total_gains': 51,
 'gain_rate': 0.6623376623376623,
 'mean_delta_Test_R2': np.float64(0.025079428571428565)}

In [8]:
# Doymuş faz: algoritma bazlı ortalama Test R² katkıları

sat_algo_summary


,Algorithm,Mean_Delta_Test_R2,N
0,AdaBoost,0.0086,7
1,CatBoost,0.0019,7
2,DART,0.0720,7
3,EF,-0.0001,7
4,ET,-0.0000,7
5,GBDT,0.0040,7
6,GP,0.1022,7
7,KNN,0.0010,7
8,LightGBM,0.0846,7
9,RF,0.0016,7


In [9]:
# Doymuş faz: hedef (çıktı) bazlı ortalama Test R² katkıları

sat_target_summary


,Target,Mean_Delta_Test_R2,N
0,P(çıktı),0.0037,11
1,h buhar (çıktı),0.0024,11
2,h sıvı (çıktı),0.0005,11
3,s buhar (çıktı),0.1771,11
4,s sıvı (çıktı),0.0007,11
5,v buhar (çıktı),0.0013,11
6,v sıvı (çıktı),-0.0102,11


In [10]:
# Kızgın faz: Base vs STGP‑EF karşılaştırması

sup_gain_analysis = analyze_stgp_ef_gains(sup_results_df)

sup_pivot = sup_gain_analysis['pivot']
sup_scenario_summary = sup_gain_analysis['scenario_summary']
sup_algo_summary = sup_gain_analysis['algo_summary']
sup_target_summary = sup_gain_analysis['target_summary']

print("Kızgın faz - Genel senaryo özeti (Base vs STGP‑EF):")
sup_scenario_summary


Kızgın faz - Genel senaryo özeti (Base vs STGP‑EF):


{'total_cases': 33,
 'total_gains': 26,
 'gain_rate': 0.7878787878787878,
 'mean_delta_Test_R2': np.float64(0.011973060606060619)}

In [11]:
# Kızgın faz: algoritma bazlı ortalama Test R² katkıları

sup_algo_summary


,Algorithm,Mean_Delta_Test_R2,N
0,AdaBoost,0.0527,3
1,CatBoost,0.0012,3
2,DART,0.0136,3
3,EF,-0.0006,3
4,ET,-0.0003,3
5,GBDT,0.0019,3
6,GP,-0.0390,3
7,KNN,0.0895,3
8,LightGBM,0.0092,3
9,RF,0.0037,3


In [12]:
# Kızgın faz: hedef (çıktı) bazlı ortalama Test R² katkıları

sup_target_summary


,Target,Mean_Delta_Test_R2,N
0,h (çıktı),0.0042,11
1,s (çıktı),0.0315,11
2,v (çıktı),0.0002,11


## 4. Hibrit Özniteliklerin Bireysel ve Toplu Katkıları

Bu aşamada, STGP‑EF ile üretilen hibrit özniteliklerin:

- Toplam başarıma katkısı (Base vs Hybrid),
- Bireysel katkıları (özellik önemi, permutation importance),

analiz edilmektedir.

Analiz, temel bir regresyon algoritması (örneğin **XGBoost**) seçilerek
gerçekleştirilir. Böylece:

- Base ve Hybrid senaryolarında,
- Her hedef değişken için,

performans metrikleri ile öznitelik önemleri elde edilir ve toplu olarak
özetlenir.


In [13]:
# Doymuş faz: hibrit öznitelik katkıları (XGBoost temel model)

sat_metrics_hybrid, sat_importances_hybrid = summarize_hybrid_contributions_for_all_targets(
    df_saturated,
    SATURATED_INPUTS,
    SATURATED_OUTPUTS,
    base_model_name='XGBoost'
)

sat_metrics_hybrid.head()


2026-05-21 23:09:29,845 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...
2026-05-21 23:09:37,249 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:09:37,250 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 23:09:37,250 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:09:37,250 - INFO -   STGP_00: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,250 - INFO -   STGP_01: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,250 - INFO -   STGP_02: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,250 - INFO -   STGP_03: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,250 - INFO -   STGP_04: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,250 - INFO -   STGP_05: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,255 - INFO -   STGP_06: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-21 23:09:37,255 - INFO -   STGP_07: (

,Scenario,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Target,Base_Model
0,Base,1.0000,0.9991,0.0021,0.0298,0.0016,0.0234,P(çıktı),XGBoost
1,Hybrid,1.0000,0.9990,0.0023,0.0322,0.0017,0.0258,P(çıktı),XGBoost
2,Base,1.0000,0.9981,0.0022,0.0439,0.0017,0.0291,v sıvı (çıktı),XGBoost
3,Hybrid,1.0000,0.9981,0.0019,0.0445,0.0014,0.0302,v sıvı (çıktı),XGBoost
4,Base,1.0000,0.9951,0.0019,0.0632,0.0014,0.0428,v buhar (çıktı),XGBoost


In [14]:
# Kızgın faz: hibrit öznitelik katkıları (XGBoost temel model)

sup_metrics_hybrid, sup_importances_hybrid = summarize_hybrid_contributions_for_all_targets(
    df_superheated,
    SUPERHEATED_INPUTS,
    SUPERHEATED_OUTPUTS,
    base_model_name='XGBoost'
)

sup_metrics_hybrid.head()


2026-05-21 23:24:29,221 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...
2026-05-21 23:24:37,226 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:24:37,232 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-21 23:24:37,232 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-21 23:24:37,232 - INFO -   STGP_00: (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0)))) - ((x1 * x1) - x0))) * (x1 * x1)))
2026-05-21 23:24:37,232 - INFO -   STGP_01: (((x1 * x1) - ((x1 + x1) + x1)) * ((x1 + (((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0))))) * (x1 * x1)))
2026-05-21 23:24:37,232 - INFO -   STGP_02: (((x1 * x1) - (x1 + (x1 + x1))) * ((x1 + (((((x1 * x1) - (x1 + x1)) * ((x1 

,Scenario,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Target,Base_Model
0,Base,1.0000,0.9981,0.0032,0.0354,0.0022,0.0151,v (çıktı),XGBoost
1,Hybrid,1.0000,0.9978,0.0017,0.0384,0.0012,0.0173,v (çıktı),XGBoost
2,Base,1.0000,0.9988,0.0061,0.0395,0.0043,0.0277,h (çıktı),XGBoost
3,Hybrid,1.0000,0.9989,0.0017,0.0364,0.0013,0.0254,h (çıktı),XGBoost
4,Base,1.0000,0.9941,0.0067,0.0672,0.0048,0.0508,s (çıktı),XGBoost


In [15]:
# Doymuş faz: Base vs Hybrid ortalama Test R² (XGBoost)

sat_metrics_hybrid.groupby('Scenario')['Test_R2'].mean()


Scenario
Base     0.9954
Hybrid   0.9954
Name: Test_R2, dtype: float64

In [16]:
# Kızgın faz: Base vs Hybrid ortalama Test R² (XGBoost)

sup_metrics_hybrid.groupby('Scenario')['Test_R2'].mean()


Scenario
Base     0.9970
Hybrid   0.9968
Name: Test_R2, dtype: float64

In [17]:
# Doymuş faz: base ve hibrit özniteliklerin ortalama önemleri

sat_importances_hybrid.groupby('type')['importance_mean'].mean()


type
base     1.7299
hybrid   0.0037
Name: importance_mean, dtype: float64

In [18]:
# Kızgın faz: base ve hibrit özniteliklerin ortalama önemleri

sup_importances_hybrid.groupby('type')['importance_mean'].mean()


type
base     0.0131
hybrid   0.0378
Name: importance_mean, dtype: float64

In [19]:
# Doymuş faz: en önemli 10 hibrit öznitelik

top_sat_hybrid_feats = (
    sat_importances_hybrid[sat_importances_hybrid['type'] == 'hybrid']
    .sort_values('importance_mean', ascending=False)
    .head(10)
)
top_sat_hybrid_feats


,feature,type,importance_mean,importance_std,Target,Base_Model
127,h2,hybrid,0.3194,0.1310,s buhar (çıktı),XGBoost
128,h0,hybrid,0.1411,0.0249,s buhar (çıktı),XGBoost
129,h3,hybrid,0.0158,0.0049,s buhar (çıktı),XGBoost
130,h1,hybrid,0.0139,0.0029,s buhar (çıktı),XGBoost
131,h11,hybrid,0.0122,0.0054,s buhar (çıktı),XGBoost
85,h10,hybrid,0.0078,0.0015,h buhar (çıktı),XGBoost
86,h15,hybrid,0.0020,0.0002,h buhar (çıktı),XGBoost
87,h14,hybrid,0.0007,0.0002,h buhar (çıktı),XGBoost
132,h18,hybrid,0.0006,0.0001,s buhar (çıktı),XGBoost
133,h17,hybrid,0.0002,0.0001,s buhar (çıktı),XGBoost


In [20]:
# Kızgın faz: en önemli 10 hibrit öznitelik

top_sup_hybrid_feats = (
    sup_importances_hybrid[sup_importances_hybrid['type'] == 'hybrid']
    .sort_values('importance_mean', ascending=False)
    .head(10)
)
top_sup_hybrid_feats


,feature,type,importance_mean,importance_std,Target,Base_Model
0,h0,hybrid,0.9009,0.0753,v (çıktı),XGBoost
22,h0,hybrid,0.3225,0.0323,h (çıktı),XGBoost
44,h13,hybrid,0.2222,0.0362,s (çıktı),XGBoost
23,h11,hybrid,0.1421,0.0147,h (çıktı),XGBoost
45,h16,hybrid,0.1088,0.0183,s (çıktı),XGBoost
46,h4,hybrid,0.1037,0.0104,s (çıktı),XGBoost
24,h1,hybrid,0.0987,0.0089,h (çıktı),XGBoost
47,h0,hybrid,0.0954,0.0141,s (çıktı),XGBoost
1,h2,hybrid,0.0893,0.0093,v (çıktı),XGBoost
48,h10,hybrid,0.0404,0.0045,s (çıktı),XGBoost


## 5. Termodinamik Tutarlılık Analizi

Bu bölümde, hem gerçek veri setleri hem de (ileride istenirse) model tahminleri
üzerinden temel termodinamik eşitsizlikler ve monotonluk koşulları test edilecektir.

### 5.1 Doymuş Faz

Doymuş faz için aşağıdaki kontroller uygulanmaktadır:

1. **Noktasal eşitsizlikler:**
   - $v_g > v_f$
   - $h_g > h_f $
   - $s_g > s_f$

2. **Monotonluk kontrolleri (T'ye göre):**
   - Doymuş basınç: $P_{\text{sat}}(T)$ monoton artan,
   - Doymuş sıvı ve buhar entalpileri,
   - Doymuş sıvı ve buhar entropileri: sıcaklıkla birlikte artan eğilim.

Bu kontroller, veri tabanının ve ileride model tahminlerinin
fiziksel tutarlılığını değerlendirmek için nicel ihlal oranları üretmektedir.


In [21]:
# Doymuş faz - Gerçek veri: noktasal eşitsizlik kontrolleri

sat_point_summary_true, sat_point_viol_true = check_saturated_pointwise_constraints(df_saturated)

print("Doymuş faz (gerçek veri) - Noktasal eşitsizlik özeti:")
sat_point_summary_true


Doymuş faz (gerçek veri) - Noktasal eşitsizlik özeti:


{'v_g > v_f': {'n_total': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)},
 'h_g > h_f': {'n_total': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)},
 's_g > s_f': {'n_total': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)}}

In [22]:
# Doymuş faz - Gerçek veri: monotonluk kontrolleri

sat_mono_summary_true = check_saturated_monotonicity(df_saturated)

print("Doymuş faz (gerçek veri) - Monotonluk özeti:")
sat_mono_summary_true


Doymuş faz (gerçek veri) - Monotonluk özeti:


{'P_vs_T_increasing': {'n_points': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)},
 'h sıvı (çıktı)_vs_T_increasing': {'n_points': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)},
 'h buhar (çıktı)_vs_T_increasing': {'n_points': 136,
  'n_violations': 16,
  'violation_rate': np.float64(0.11851851851851852)},
 's sıvı (çıktı)_vs_T_increasing': {'n_points': 136,
  'n_violations': 0,
  'violation_rate': np.float64(0.0)},
 's buhar (çıktı)_vs_T_increasing': {'n_points': 136,
  'n_violations': 63,
  'violation_rate': np.float64(0.4666666666666667)}}

### 5.2 Kızgın Faz

Kızgın faz (superheated) bölgesi için ise:

1. Sabit basınçta:
   - \( v(T) \), \( h(T) \), \( s(T) \) fonksiyonlarının artan olması beklenmektedir.

2. Sabit sıcaklıkta:
   - \( v(P) \) fonksiyonunun genel olarak azalan olması beklenmektedir
     (aynı sıcaklıkta basınç arttıkça özgül hacmin azalması).

Aşağıda, bu koşulların sağlanma durumu ortalama ihlal oranları üzerinden
nicel olarak raporlanmaktadır.


In [23]:
# Kızgın faz - Gerçek veri: monotonluk kontrolleri

sup_mono_summary_true = check_superheated_monotonicity(df_superheated)

print("Kızgın faz (gerçek veri) - Monotonluk özeti:")
sup_mono_summary_true


Kızgın faz (gerçek veri) - Monotonluk özeti:


{'v (çıktı)_vs_T_at_constP_increasing': np.float64(0.0),
 'h (çıktı)_vs_T_at_constP_increasing': np.float64(0.0),
 's (çıktı)_vs_T_at_constP_increasing': np.float64(0.0),
 'v (çıktı)_vs_P_at_constT_decreasing': np.float64(0.0)}

> Not: Aynı denetimler, Base ve STGP‑EF senaryoları altında elde edilen model
> tahminleri ile oluşturulacak veri tablolarına da uygulanabilir. Böylece,
> yüksek Test R² değerlerinin yanı sıra, modellerin termodinamik yasalarla
> uyumu da karşılaştırmalı olarak değerlendirilebilir.


## 6. Genel Değerlendirme

Bu notebook kapsamında:

1. R515B soğutucu akışkanının doymuş ve kızgın faz verileri üzerinde,
   girdi uzayı **STGP‑EF hibrit yöntemi** ile zenginleştirilmiş,

2. 11 farklı regresyon algoritması için:
   - Base ve STGP‑EF senaryolarında elde edilen performans metrikleri,
   - Her (hedef, algoritma) çifti için Test R² kazanımları,
   - Algoritma ve hedef bazında ortalama katkılar

nicel olarak ortaya konulmuştur.

3. STGP‑EF ile üretilen hibrit özniteliklerin:
   - Bireysel önemleri (permutation importance),
   - Base özniteliklere göre göreli katkıları,

doymuş ve kızgın fazlar için ayrı ayrı analiz edilmiştir.

4. Son olarak, hem veri setleri hem de (uygulandığı takdirde) model
tahminleri üzerinde temel termodinamik tutarlılık kontrolleri yapılarak,
yüksek doğruluk ile fiziksel tutarlılık arasındaki etkileşim tartışmaya
açılmıştır.

Bu çerçevede, STGP‑EF yaklaşımının, sınırlı girdili termodinamik veri
setlerinde hem tahmin doğruluğunu artırma hem de fiziksel tutarlılık
bağlamında denetlenebilirlik sağlama açısından güçlü bir araç olduğu
görülmektedir.
